# Train "KINO" Wake-Word Model for LangRobo

**KINO** = *Knowledge Is Never Out*

This notebook automates training the custom acoustic wake-word model for **"Kino"** / **"Hey Kino"** using [openWakeWord](https://github.com/dscripka/openWakeWord).

### What this notebook does:
1. Sets up the Linux + Piper TTS training environment with GPU acceleration
2. Synthesizes thousands of varied acoustic samples of "Kino", "Hey Kino", "Hi Kino"
3. Mixes in targeted adversarial negative phrases (`casino`, `keynote`, `you know`, `we know`, `kilo`, `cinema`, `quinoa`, `rhino`, `dino`, etc.) to prevent false wakes
4. Augments samples with Room Impulse Responses (RIRs) and background noise
5. Trains a binary classifier against 2,000+ hours of speech background features
6. Exports **`kino.onnx`** (~414 KB) and downloads it directly to your machine

**Runtime:** ~20–30 minutes on a free Google Colab T4 GPU.
**Instructions:** In Colab menu, select **Runtime > Change runtime type > T4 GPU**, then click **Runtime > Run all**.

## Step 1: Environment Setup
Installs openWakeWord, piper-sample-generator, dependencies, and base feature models.

In [ ]:
# Install system dependencies
!apt-get update -qq && apt-get install -y libsndfile1 ffmpeg -qq

# Clone piper-sample-generator for synthetic speech generation
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
    https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt

!pip install -q piper-phonemize webrtcvad soundfile

# Clone openwakeword and install dependencies
!git clone https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword
!pip install -q mutagen torchinfo torchmetrics speechbrain audiomentations torch-audiomentations acoustics pronouncing datasets deep-phonemizer onnxruntime

# Ensure base embedding and melspectrogram models are downloaded
import os, openwakeword
models_dir = os.path.join(os.path.dirname(openwakeword.__file__), "resources", "models")
os.makedirs(models_dir, exist_ok=True)

!wget -q -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -P {models_dir}
!wget -q -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -P {models_dir}
print("Environment and base models ready!")

## Step 2: Download Room Impulse Responses (RIRs) & Background Features
Downloads pre-computed features from ACAV100M (~2,000 hours of real background speech/audio) and MIT environmental impulse responses for realistic room acoustics.

In [ ]:
# Download MIT RIRs
import os, datasets
output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)
print("Downloading Room Impulse Responses...")
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in rir_dataset.take(500):
    with open(os.path.join(output_dir, row["audio"]["path"]), "wb") as f:
        f.write(row["audio"]["bytes"])

# Download pre-computed negative features (~2,000 hrs ACAV100M subset) & validation set
print("Downloading pre-computed negative feature sets...")
!wget -q -nc https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q -nc https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print("Background datasets downloaded!")

## Step 3: Configure "KINO" Training Parameters
Configures the target phrases (`kino`, `hey kino`, `hi kino`) and hard adversarial words (`keynote`, `casino`, `you know`, `we know`, `kilo`, `cinema`, etc.).

In [ ]:
import yaml

config_yaml = """
model_name: "kino"

target_phrase:
  - "kino"
  - "hey kino"
  - "hi kino"
  - "hello kino"

custom_negative_phrases:
  - "keynote"
  - "casino"
  - "you know"
  - "he know"
  - "we know"
  - "kimono"
  - "quinoa"
  - "kilo"
  - "cinema"
  - "mosquito"
  - "keanu"
  - "rhino"
  - "dino"
  - "cleaner"

n_samples: 5000
n_samples_val: 1000
tts_batch_size: 50
augmentation_batch_size: 16
piper_sample_generator_path: "./piper-sample-generator"
output_dir: "./kino_training"

rir_paths:
  - "./mit_rirs"
background_paths: []
background_paths_duplication_rate: []

false_positive_validation_data_path: "./validation_set_features.npy"
augmentation_rounds: 1

feature_data_files:
  "ACAV100M_sample": "./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"

batch_n_per_class:
  "ACAV100M_sample": 1024
  "adversarial_negative": 50
  "positive": 50

model_type: "dnn"
layer_size: 32
steps: 15000
max_negative_weight: 1500
target_false_positives_per_hour: 0.2
"""

with open("kino_config.yaml", "w") as f:
    f.write(config_yaml)

print("Saved kino_config.yaml")

## Step 4: Generate Synthetic Clips, Augment & Train
Generates audio samples for KINO and negatives, augments them with acoustic room models, and trains the neural network.

In [ ]:
import sys

# 1. Generate synthetic positive and adversarial negative clips with Piper TTS
print("=== Step 4a: Generating synthetic clips with Piper TTS ===")
!{sys.executable} openwakeword/openwakeword/train.py --training_config kino_config.yaml --generate_clips

# 2. Augment clips (room reverberation, noise, pitch/speed)
print("=== Step 4b: Augmenting clips ===")
!{sys.executable} openwakeword/openwakeword/train.py --training_config kino_config.yaml --augment_clips

# 3. Train the model and export ONNX
print("=== Step 4c: Training model against background features ===")
!{sys.executable} openwakeword/openwakeword/train.py --training_config kino_config.yaml --train_model


## Step 5: Verify and Download `kino.onnx`
Locates the trained `kino.onnx` model and triggers a direct browser download.

In [ ]:
import glob, shutil
from google.colab import files

# Find generated onnx model
candidates = glob.glob("./kino_training/**/*.onnx", recursive=True)
if not candidates:
    candidates = glob.glob("./**/*kino*.onnx", recursive=True)

if candidates:
    target_onnx = candidates[0]
    dest = "kino.onnx"
    shutil.copy(target_onnx, dest)
    print(f"Found model at {target_onnx}! Ready for download.")
    files.download(dest)
else:
    print("Model file not found. Check training logs above for any errors.")

## Next Steps (Deploy to Robot)

Once `kino.onnx` is downloaded to your machine:

1. Copy `kino.onnx` into `src/langrobo_ros/models/wake/kino.onnx` in this repo:
   ```bash
   cp ~/Downloads/kino.onnx src/langrobo_ros/models/wake/kino.onnx
   ```

2. Test it locally against microphone or recorded clips:
   ```bash
   python3 scripts/wake_live_score.py src/langrobo_ros/models/wake/kino.onnx --threshold 0.45
   ```

3. Switch the robot wake gate in one command:
   ```bash
   ./scripts/wake_switch.py kino --threshold 0.45
   ```